# Phase 6 — MPRA activity prediction with locus-held-out evaluation

**Kaggle settings:** GPU (T4/P100 or better), Internet ON. Run cells in order. This is a new task: predicting the top versus bottom within-design activity quintiles in HepG2 and K562, not repeating Stage 1 enhancer classification. No Phase 4 labels were used in the Phase 5 checkpoint.

**Attach exactly these two completed outputs as Kaggle Inputs (private datasets):**

1. `4_enhancer_phase4_external_dataset.zip` (Phase 4; 30,340 labeled 200-bp sequences).
2. `enhancer_phase5_complete_with_weights.zip` (Phase 5 recovery; original results ZIP and both fine-tuned Stage 1 `.pt` checkpoints).

Kaggle may show each ZIP or its unpacked members; the loader accepts either. The notebook checks member checksums. It makes one deterministic, shared train/validation/test split over overlapping genomic loci in **both** cell lines; exact or reverse-complement sequence matches across loci are kept in one split. It fits each cell separately. Model selection uses validation AUROC; the test set is scored only after selection. Conventional GC, 3-mer and spaced-6/4-mer models are compared with frozen Phase 5 transfer scores, MPRA-fine-tuned DNABERT-2 from pretrained weights, MPRA-fine-tuned DNABERT-2 initialized from Phase 5, and similarly initialized legacy DNABERT. A fresh classification head is used for every MPRA fine-tune. All trainable models include reverse-complement augmentation only on training rows.

**Outputs:** `/kaggle/working/enhancer_phase6_mpra_results.zip` (split, scores, validation choices, metrics, grouped bootstrap, provenance) and `/kaggle/working/enhancer_phase6_complete_with_weights.zip` (all results and the **six** fitted model checkpoints). Download both or at minimum the complete ZIP. The complete ZIP is several GB. Save the Kaggle notebook version with outputs for a durable copy. For browser links use `os.chdir('/kaggle/working'); display(FileLink('enhancer_phase6_complete_with_weights.zip'))`.

This single holdout is an external-assay extension; it does not replace the original grouped iEnhancer benchmark. Read the test metrics only after the run is complete. If a session ends before packaging, attach its saved Kaggle Output as an Input on the next run to resume, including `enhancer_phase6_work/checkpoints` and `enhancer_phase6_work/predictions`.


In [1]:
# 0 — Dependencies, frozen configuration and portable helpers.
import os, sys, io, json, zipfile, hashlib, random, gc, shutil, warnings, subprocess, time
from pathlib import Path
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import spearmanr
from itertools import combinations
ROOT=Path('/kaggle/input'); assert ROOT.is_dir(),'Attach the two ZIPs as Kaggle Inputs.'
WORK=Path('/kaggle/working/enhancer_phase6_work');WORK.mkdir(parents=True,exist_ok=True)
PRED=WORK/'predictions';PRED.mkdir(exist_ok=True)
CKPT=WORK/'checkpoints';CKPT.mkdir(exist_ok=True)
EXPECTED={'phase4_member_sha':'b0bf4a433f1e0af74d95b8747b9312538734f70c425c8379628cafef0c947d2a',
          'phase4_protocol':'e49fe4d57dda904bffe9fa5a47d8c8e397668cbee71f24e3b0152fd1781b8705',
          'phase5_protocol':'4a1e7bead22829b5ea866bceaf80fc02d174cb18c2385040a764f65eec9bdee4'}
REVISIONS={'dnabert_legacy':'c56e67ea5827e0ddc67ef059addcf71569b1216e',
           'dnabert2':'ec1f874253852eb3907081f57294991b4280ceb6'}
MODEL_IDS={'dnabert_legacy':'zhihan1996/DNA_bert_6','dnabert2':'zhihan1996/DNABERT-2-117M'}
ARMS=['gc_logreg','k3_logreg','spaced6_4_logreg','dnabert2_pretrained',
      'dnabert2_stage1_init','dnabert_legacy_stage1_init']
CONFIG={'seed':20260929,'test_fraction_groups':.20,'validation_fraction_remaining_groups':.1875,
        'epochs':2,'batch':8,'accumulation':2,'lr':2e-5,'weight_decay':.01,
        'max_tokens':256,'inference_batch':32,'bootstrap_repeats':300,
        'classical_C_candidates':[.1,1.,10.], 'arms':ARMS,
        'phase4_protocol':EXPECTED['phase4_protocol'],'phase5_protocol':EXPECTED['phase5_protocol'],
        'revisions':REVISIONS,'version':'phase6_mpra_locus_holdout_v1'}
def sha(raw):return hashlib.sha256(raw).hexdigest()
def sha_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
# Import earlier partial Phase 6 notebook Output, if attached, to resume an interrupted run.
for folder,kind in [(PRED,'predictions'),(CKPT,'checkpoints')]:
 previous=[p for p in ROOT.rglob('*') if p.is_file() and
           p.parent.name==kind and p.parent.parent.name=='enhancer_phase6_work']
 for name in sorted({p.name for p in previous}):
  choices=[p for p in previous if p.name==name]
  assert len({sha_file(p) for p in choices})==1,f'Ambiguous prior Phase 6 cache: {name}'
  target=folder/name
  if target.exists():
   assert sha_file(target)==sha_file(choices[0]),f'Conflicting active cache: {name}'
  else:
   shutil.copy2(choices[0],target)
 if previous:print('Imported',len(previous),'earlier Phase 6 cache files from Kaggle Input.')
RC=str.maketrans('ACGT','TGCA')
def revcomp(s):return s.translate(RC)[::-1]
def locate_zip(name,required_member):
 hits=[]
 for root in [ROOT,Path('/kaggle/working')]:
  hits.extend(p for p in root.rglob(name) if p.is_file() and WORK not in p.parents)
 good=[]
 for p in hits:
  with zipfile.ZipFile(p) as z:
   if required_member in z.namelist():good.append(p)
 if not good:
  unpacked=[p for p in ROOT.rglob(required_member) if p.is_file()]
  assert len(unpacked)==1,f'Attach one {name} ZIP or its unpacked output; found {hits}; unpacked {unpacked}'
  base=unpacked[0].parent
  if name.startswith('4_'):
   members=['phase4_external_relative_activity_all_cells.csv','phase4_audit_and_provenance.json']
  else:
   manifest=json.loads(unpacked[0].read_text())
   members=[required_member,manifest['original_results_filename']]+[
    x['filename'] for x in manifest['checkpoints'].values()]
  normalized=WORK/'normalized_inputs';normalized.mkdir(exist_ok=True)
  path=normalized/name
  for m in members:assert (base/m).is_file(),f'Missing unpacked archive member: {base/m}'
  with zipfile.ZipFile(path,'w',allowZip64=True) as z:
   for m in members:z.write(base/m,m,compress_type=zipfile.ZIP_STORED)
  print('Repacked Kaggle-unpacked Input:',name)
  return path
 if len(good)>1:
  unique={sha_file(p) for p in good}
  assert len(unique)==1,f'Ambiguous versions of {name}: {good}'
 return sorted(good,key=lambda p: str(p).startswith('/kaggle/input'),reverse=True)[0]
def member_hash(z,name):
 h=hashlib.sha256()
 with z.open(name) as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
print('Protocol',LOCK)


Protocol 6f11a0b1ebbb74ea5616de383b641385a272691f8b22bc32fd25284fd984f7af


In [2]:
# 1 — Read and authenticate Phase 4 and the Phase 5 complete bundle.
P4 = locate_zip(
    '4_enhancer_phase4_external_dataset.zip',
    'phase4_external_relative_activity_all_cells.csv'
)
P5 = locate_zip(
    'enhancer_phase5_complete_with_weights.zip',
    'phase5_complete_bundle_manifest.json'
)

with zipfile.ZipFile(P4) as z:
    assert z.testzip() is None
    audit = json.loads(z.read('phase4_audit_and_provenance.json'))
    assert audit['protocol_sha256'] == EXPECTED['phase4_protocol']

    name = 'phase4_external_relative_activity_all_cells.csv'
    assert (
        audit['output_sha256'][name]
        == EXPECTED['phase4_member_sha']
        == member_hash(z, name)
    )
    d = pd.read_csv(
        io.BytesIO(z.read(name)),
        dtype={
            'record_id': str,
            'sequence': str,
            'sequence_sha256': str,
            'locus_group_id': str,
            'chrom': str,
            'design_name': str,
        }
    )

with zipfile.ZipFile(P5) as z:
    assert z.testzip() is None
    bundle = json.loads(z.read('phase5_complete_bundle_manifest.json'))
    assert bundle['phase5_protocol_hash'] == EXPECTED['phase5_protocol']
    assert set(bundle['checkpoints']) == set(REVISIONS)

    for key, meta in bundle['checkpoints'].items():
        assert meta['revision'] == REVISIONS[key]
        assert member_hash(z, meta['filename']) == meta['sha256'], key

    results_name = bundle['original_results_filename']
    assert member_hash(z, results_name) == bundle['original_results_sha256']

    with zipfile.ZipFile(io.BytesIO(z.read(results_name))) as inner:
        phase5_provenance = json.loads(
            inner.read('phase5_provenance.json')
        )
        assert (
            phase5_provenance['protocol_hash']
            == EXPECTED['phase5_protocol']
        )

        for name, digest in phase5_provenance['output_sha256'].items():
            assert sha(inner.read(name)) == digest, name

        transfer = pd.read_csv(
            io.BytesIO(inner.read('phase5_external_predictions.csv')),
            dtype={
                'record_id': str,
                'cell_line': str,
                'model': str,
                'locus_group_id': str,
            }
        )

assert len(d) == 30340 and d.record_id.is_unique
assert d.cell_line.value_counts().to_dict() == {
    'K562': 17198, 'HepG2': 13142
}
assert d.sequence.str.fullmatch('[ACGT]{200}').all()
assert all(
    sha(s.encode()) == h
    for s, h in zip(d.sequence, d.sequence_sha256)
)
assert d.groupby(['chrom', 'start', 'end']).locus_group_id.nunique().max() == 1
assert set(d.label) == {0, 1}

expected_transfer_models = {
    'gc_logreg',
    'k3_logreg',
    'spaced6_4_logreg',
    'frozen_gena_lm',
    'finetuned_dnabert_legacy',
    'finetuned_dnabert2',
    'raw_gc',
}
assert set(transfer.model) == expected_transfer_models
assert len(transfer) == len(d) * len(expected_transfer_models)
assert transfer.protocol_hash.eq(EXPECTED['phase5_protocol']).all()
assert not transfer.duplicated(['model', 'record_id']).any()

for model, part in transfer.groupby('model'):
    assert len(part) == len(d), model
    assert set(part.record_id) == set(d.record_id), model

print(
    'VERIFIED', len(d),
    'MPRA rows, six Phase 5 models and raw-GC diagnostic; '
    'source checkpoints authenticated.'
)

Repacked Kaggle-unpacked Input: 4_enhancer_phase4_external_dataset.zip
VERIFIED 30340 MPRA rows, six Phase 5 models and raw-GC diagnostic; source checkpoints authenticated.


In [3]:
# 2 — One shared group split, with exact and reverse-complement matches joined to their loci.
parent={g:g for g in d.locus_group_id.unique()}
def find(g):
 while parent[g]!=g:
  parent[g]=parent[parent[g]];g=parent[g]
 return g
def union(a,b):
 a,b=find(a),find(b)
 if a!=b:parent[max(a,b)]=min(a,b)
seen={}
for row in d[['sequence','locus_group_id']].itertuples(index=False):
 key=min(row.sequence,revcomp(row.sequence))
 if key in seen:union(row.locus_group_id,seen[key])
 else:seen[key]=row.locus_group_id
d['split_group_id']=d.locus_group_id.map(find)
groups=np.array(sorted(d.split_group_id.unique()))
gss=GroupShuffleSplit(n_splits=1,test_size=CONFIG['test_fraction_groups'],random_state=CONFIG['seed'])
remaining,test=next(gss.split(groups,groups=groups))
remain_groups=groups[remaining];test_groups=set(groups[test])
val_split=GroupShuffleSplit(n_splits=1,test_size=CONFIG['validation_fraction_remaining_groups'],random_state=CONFIG['seed']+1)
train_idx,val_idx=next(val_split.split(remain_groups,groups=remain_groups))
val_groups=set(remain_groups[val_idx]);train_groups=set(remain_groups[train_idx])
assert not ((test_groups & val_groups)|(val_groups & train_groups)|(test_groups & train_groups))
d['split']=np.where(d.split_group_id.isin(test_groups),'test',
             np.where(d.split_group_id.isin(val_groups),'validation','train'))
assert set(d.split)=={'train','validation','test'}
assert d.groupby('locus_group_id').split.nunique().max()==1
assert d.groupby(d.sequence.map(lambda s:min(s,revcomp(s)))).split.nunique().max()==1
counts=d.groupby(['cell_line','split','label']).size().unstack(fill_value=0)
assert (counts>0).all().all(),counts
assert (counts.sum(axis=1)>100).all(),counts
split_file=WORK/'phase6_split.csv'
d[['record_id','cell_line','locus_group_id','split_group_id','split','label']].to_csv(split_file,index=False)
print(counts)
print('Locus components:',len(groups),'split hash:',sha_file(split_file))


label                    0     1
cell_line split                 
HepG2     test        1287  1352
          train       4189  4327
          validation   983  1004
K562      test        1818  1630
          train       5912  5265
          validation  1365  1208
Locus components: 28115 split hash: 75fd23cc8f223dae37b6d4c68266750c933699fe187335103a7164a6ceb193f4


In [4]:
# 3 — Conventional baselines: fitting and C selection on training/validation only.
def seeds(seed):
 random.seed(seed);np.random.seed(seed)
 if 'torch' in sys.modules:
  import torch
  torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def gc_features(seq):return np.array([[(s.count('G')+s.count('C'))/200,
                                        s.count('CG')/199] for s in seq],dtype=np.float32)
def tri_tokens(s):return [s[i:i+3] for i in range(198)]
MASKS=tuple(combinations(range(6),4))
def spaced_tokens(s):return [str(mi)+''.join(s[i+j] for j in mask)
                             for i in range(195) for mi,mask in enumerate(MASKS)]
SCORES=[];CHOICES=[]
def emit(cell,arm,frame,scores):
 v=np.asarray(scores,dtype=float)
 assert len(v)==len(frame) and np.isfinite(v).all() and ((v>=0)&(v<=1)).all()
 SCORES.extend({'record_id':rid,'cell_line':cell,'split':part,'label':int(label),
                'locus_group_id':locus,'model':arm,'score':float(score)}
               for rid,part,label,locus,score in zip(frame.record_id,frame.split,frame.label,
                                                     frame.locus_group_id,v))
def fit_classical(cell,arm):
 sub=d[d.cell_line==cell].copy()
 tr=sub[sub.split=='train'];va=sub[sub.split=='validation'];te=sub[sub.split=='test']
 cache=PRED/f'{cell}_{arm}_classical.csv';lockfile=PRED/f'{cell}_{arm}_classical_lock.json'
 if cache.is_file() and lockfile.is_file():
  meta=json.loads(lockfile.read_text())
  if meta['protocol']==LOCK and sha_file(cache)==meta['sha256']:
   old=pd.read_csv(cache,dtype={'record_id':str,'split':str,'score':float})
   expected=pd.concat([va,te])
   assert old.record_id.tolist()==expected.record_id.tolist()
   assert old.split.tolist()==expected.split.tolist()
   CHOICES.append(meta['choice'])
   emit(cell,arm,expected,old.score.to_numpy())
   print('REUSED',cell,arm,flush=True)
   return
 ta=tr.sequence.tolist();ta+=list(map(revcomp,tr.sequence.tolist()))
 y=np.tile(tr.label.to_numpy(dtype=int),2)
 if arm=='gc_logreg':
  X=gc_features(ta);V=gc_features(va.sequence);T=gc_features(te.sequence)
  def fit(c):return make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=1000)).fit(X,y)
 elif arm=='k3_logreg':
  vec=CountVectorizer(analyzer=tri_tokens,lowercase=False,dtype=np.float32)
  X=vec.fit_transform(ta);V=vec.transform(va.sequence);T=vec.transform(te.sequence)
  def fit(c):return make_pipeline(Normalizer(norm='l1'),LogisticRegression(C=c,max_iter=1000,solver='liblinear')).fit(X,y)
 elif arm=='spaced6_4_logreg':
  vec=TfidfVectorizer(analyzer=spaced_tokens,lowercase=False,dtype=np.float32,norm='l2')
  X=vec.fit_transform(ta);V=vec.transform(va.sequence);T=vec.transform(te.sequence)
  def fit(c):return LogisticRegression(C=c,max_iter=1000,solver='liblinear').fit(X,y)
 else:raise ValueError(arm)
 results=[]
 with warnings.catch_warnings():
  warnings.filterwarnings('ignore',category=ConvergenceWarning)
  for c in CONFIG['classical_C_candidates']:
   m=fit(c);pv=m.predict_proba(V)[:,1]
   results.append((roc_auc_score(va.label,pv),c,m,pv))
 best=max(results,key=lambda x:(x[0],-x[1]))
 choice={'cell_line':cell,'model':arm,'selected_C':float(best[1]),
         'selected_epoch':'','validation_auroc':float(best[0])}
 CHOICES.append(choice)
 testprob=best[2].predict_proba(T)[:,1]
 emit(cell,arm,va,best[3]);emit(cell,arm,te,testprob)
 recorded=pd.DataFrame({'record_id':pd.concat([va,te]).record_id,
                        'split':pd.concat([va,te]).split,
                        'score':np.r_[best[3],testprob]})
 temp=cache.with_suffix('.tmp');recorded.to_csv(temp,index=False,float_format='%.12g');os.replace(temp,cache)
 lockfile.write_text(json.dumps({'protocol':LOCK,'sha256':sha_file(cache),'choice':choice},sort_keys=True))
 print(cell,arm,'C',best[1],'validation AUROC',round(best[0],4),flush=True)
 del X,V,T,results,ta
 gc.collect()
for cell in ('HepG2','K562'):
 for arm in ARMS[:3]:fit_classical(cell,arm)


REUSED HepG2 gc_logreg
REUSED HepG2 k3_logreg
REUSED HepG2 spaced6_4_logreg
REUSED K562 gc_logreg
REUSED K562 k3_logreg
REUSED K562 spaced6_4_logreg


In [5]:
# 4 — Matched Phase 5 transfer scores on this notebook's held-out rows.
for model in sorted(transfer.model.unique()):
 for cell in ('HepG2','K562'):
  relevant=d[(d.cell_line==cell)&(d.split!='train')]
  old=transfer[(transfer.cell_line==cell)&(transfer.model==model)].set_index('record_id')
  assert old.index.is_unique and set(relevant.record_id)<=set(old.index)
  matched=old.loc[relevant.record_id]
  assert matched.locus_group_id.tolist()==relevant.locus_group_id.tolist()
  assert matched.label.to_numpy(dtype=int).tolist()==relevant.label.to_numpy(dtype=int).tolist()
  emit(cell,'phase5_zero_shot_'+model,relevant,matched.score.to_numpy())
print('Reused Phase 5 zero-shot predictions with exact row identity checks.')


Reused Phase 5 zero-shot predictions with exact row identity checks.


In [6]:
# 5 — Pinned encoders, checkpoint loading, and train/validation selection.
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
assert torch.cuda.is_available(),'Select a Kaggle GPU accelerator.'
DEVICE=torch.device('cuda:0')
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
try:torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:pass
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!='4.44.2':
 assert 'transformers' not in sys.modules,'Restart session: Transformers imported before pinning.'
 subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2'],check=True,timeout=900)
from transformers import AutoTokenizer,AutoModel
import transformers
assert transformers.__version__=='4.44.2'
def tokenizer_and_encoder(key):
 remote=MODEL_IDS[key];rev=REVISIONS[key]
 tok=AutoTokenizer.from_pretrained(remote,revision=rev,trust_remote_code=True)
 if key=='dnabert_legacy':
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=rev)
  enc=AutoModel.from_pretrained(remote,revision=rev,trust_remote_code=True,config=cfg)
 else:
  try:enc=AutoModel.from_pretrained(remote,revision=rev,trust_remote_code=True)
  except (ValueError,TypeError):
   from transformers.models.bert.configuration_bert import BertConfig
   cfg=BertConfig.from_pretrained(remote,revision=rev)
   enc=AutoModel.from_pretrained(remote,revision=rev,trust_remote_code=True,config=cfg)
 if key=='dnabert2':
  import importlib
  mod=importlib.import_module(type(enc).__module__)
  assert hasattr(mod,'BertUnpadSelfAttention') and hasattr(mod,'flash_attn_qkvpacked_func')
  mod.flash_attn_qkvpacked_func=None
 return tok,enc
def tokenize(tok,key,seqs):
 values=[' '.join(s[i:i+6] for i in range(195)) if key=='dnabert_legacy' else s for s in seqs]
 b=tok(values,padding=True,add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['max_tokens'],(key,b['input_ids'].shape)
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}
def pooled(enc,b):
 out=enc(**b);h=getattr(out,'last_hidden_state',None)
 if h is None:h=out[0]
 assert h.shape[:2]==b['input_ids'].shape
 mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
 return (h*mask).sum(1)/mask.sum(1).clamp(min=1)
class BinaryEncoder(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc
  dim=int(enc.config.hidden_size)
  self.dropout=nn.Dropout(.1);self.head=nn.Linear(dim,2)
 def forward(self,b):return self.head(self.dropout(pooled(self.encoder,b)))
def source_encoder(net,key):
 name=bundle['checkpoints'][key]['filename']
 local=WORK/name
 if not local.exists():
  with zipfile.ZipFile(P5) as z:
   with z.open(name) as src,open(local.with_suffix('.part'),'wb') as dst:shutil.copyfileobj(src,dst,8*1024*1024)
  os.replace(local.with_suffix('.part'),local)
 assert sha_file(local)==bundle['checkpoints'][key]['sha256']
 saved=torch.load(local,map_location='cpu',weights_only=True)
 assert saved['lock']==EXPECTED['phase5_protocol'] and saved['revision']==REVISIONS[key]
 state={k.removeprefix('encoder.'):v for k,v in saved['state_dict'].items() if k.startswith('encoder.')}
 net.encoder.load_state_dict(state,strict=True)
 # Never load source classification head: the MPRA outcome is different.
 del saved,state
def predict_net(net,tok,key,seqs):
 net.eval();out=[]
 with torch.inference_mode():
  for start in range(0,len(seqs),CONFIG['inference_batch']):
   b=tokenize(tok,key,seqs[start:start+CONFIG['inference_batch']])
   with torch.autocast('cuda',dtype=torch.float16):
    out.extend(torch.softmax(net(b).float(),dim=1)[:,1].cpu().numpy())
 return np.asarray(out)
def fit_deep(cell,arm):
 key='dnabert_legacy' if arm.startswith('dnabert_legacy') else 'dnabert2'
 tr=d[(d.cell_line==cell)&(d.split=='train')]
 va=d[(d.cell_line==cell)&(d.split=='validation')]
 te=d[(d.cell_line==cell)&(d.split=='test')]
 class Rows(Dataset):
  def __len__(self):return 2*len(tr)
  def __getitem__(self,i):
   row=tr.iloc[i%len(tr)]
   return revcomp(row.sequence) if i>=len(tr) else row.sequence,int(row.label)
 def collate(rows):return tokenize(tok,key,[s for s,_ in rows]),torch.tensor([y for _,y in rows],dtype=torch.long,device=DEVICE)
 ckpt=CKPT/f'{cell}_{arm}_best.pt'
 valfile=PRED/f'{cell}_{arm}_validation.csv'
 complete=PRED/f'{cell}_{arm}_complete.json'
 scorefile=PRED/f'{cell}_{arm}_deep_scores.csv'
 if complete.exists() and scorefile.exists() and ckpt.exists():
  meta=json.loads(complete.read_text())
  assert meta['protocol']==LOCK and sha_file(ckpt)==meta['checkpoint_sha256']
  assert sha_file(scorefile)==meta['score_sha256']
  old=pd.read_csv(scorefile,dtype={'record_id':str,'split':str,'score':float})
  expected=pd.concat([va,te])
  assert old.record_id.tolist()==expected.record_id.tolist()
  assert old.split.tolist()==expected.split.tolist()
  CHOICES.append(meta['choice'])
  emit(cell,arm,expected,old.score.to_numpy())
  print('REUSED complete',cell,arm,flush=True)
  return
 seeds(CONFIG['seed']+17+ARMS.index(arm)+(['HepG2','K562'].index(cell)*100))
 tok,enc=tokenizer_and_encoder(key);net=BinaryEncoder(enc).to(DEVICE)
 if 'stage1_init' in arm:source_encoder(net,key)
 # Initialization above leaves the fresh MPRA head untouched.
 loader=DataLoader(Rows(),batch_size=CONFIG['batch'],shuffle=True,num_workers=0,
   collate_fn=collate,generator=torch.Generator().manual_seed(CONFIG['seed']+91+ARMS.index(arm)))
 opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['lr'],weight_decay=CONFIG['weight_decay'])
 scaler=torch.amp.GradScaler('cuda',enabled=True)
 best_auc=-1.;epoch=0;valpred=None
 for ep in range(1,CONFIG['epochs']+1):
  net.train();opt.zero_grad(set_to_none=True);losses=[]
  for step,(b,y) in enumerate(loader):
   with torch.autocast('cuda',dtype=torch.float16):loss=nn.functional.cross_entropy(net(b),y)
   assert torch.isfinite(loss),(cell,arm,ep,step)
   scaler.scale(loss/CONFIG['accumulation']).backward()
   if (step+1)%CONFIG['accumulation']==0 or step+1==len(loader):
    scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),1.0)
    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
   losses.append(float(loss.detach().cpu()))
  pv=predict_net(net,tok,key,va.sequence.tolist())
  auc=roc_auc_score(va.label,pv)
  print(cell,arm,'epoch',ep,'mean loss',round(float(np.mean(losses)),4),
        'validation AUROC',round(auc,4),flush=True)
  if auc>best_auc:
   best_auc=auc;epoch=ep;valpred=pv
   part=ckpt.with_suffix('.part')
   torch.save({'protocol':LOCK,'cell':cell,'arm':arm,'revision':REVISIONS[key],
               'epoch':ep,'validation_auroc':float(auc),
               'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()}},part)
   os.replace(part,ckpt)
   temp=valfile.with_suffix('.tmp')
   pd.DataFrame({'record_id':va.record_id,'score':pv}).to_csv(temp,index=False,float_format='%.12g')
   os.replace(temp,valfile)
 del loader,opt,scaler
 del net
 gc.collect();torch.cuda.empty_cache()
 tok,enc=tokenizer_and_encoder(key);net=BinaryEncoder(enc).to(DEVICE)
 saved=torch.load(ckpt,map_location='cpu',weights_only=True)
 net.load_state_dict(saved['state_dict'],strict=True)
 choice={'cell_line':cell,'model':arm,'selected_C':'',
         'selected_epoch':int(epoch),'validation_auroc':float(best_auc)}
 CHOICES.append(choice)
 emit(cell,arm,va,valpred)
 testpred=predict_net(net,tok,key,te.sequence.tolist())
 emit(cell,arm,te,testpred)
 frame=pd.DataFrame({'record_id':pd.concat([va,te]).record_id,
                     'split':pd.concat([va,te]).split,
                     'score':np.r_[valpred,testpred]})
 temp=scorefile.with_suffix('.tmp');frame.to_csv(temp,index=False,float_format='%.12g');os.replace(temp,scorefile)
 complete.write_text(json.dumps({'protocol':LOCK,'checkpoint_sha256':sha_file(ckpt),
                                  'score_sha256':sha_file(scorefile),'choice':choice},sort_keys=True))
 del net,enc,tok,saved
 gc.collect();torch.cuda.empty_cache()
for cell in ('HepG2','K562'):
 for arm in ARMS[3:]:fit_deep(cell,arm)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.3 MB/s eta 0:00:00
REUSED complete HepG2 dnabert2_pretrained
REUSED complete HepG2 dnabert2_stage1_init
REUSED complete HepG2 dnabert_legacy_stage1_init
REUSED complete K562 dnabert2_pretrained
REUSED complete K562 dnabert2_stage1_init
REUSED complete K562 dnabert_legacy_stage1_init


In [7]:
# 6 — Test metrics, exact GC diagnostic and paired locus bootstrap.
pred=pd.DataFrame(SCORES)
assert not pred.duplicated(['cell_line','model','record_id']).any()
for cell in ('HepG2','K562'):
 for model in pred.model.unique():
  sub=pred[(pred.cell_line==cell)&(pred.model==model)]
  assert set(sub.record_id)==set(d[(d.cell_line==cell)&(d.split!='train')].record_id)
truth=d[['record_id','cell_line','split','label','mean_log2_activity','sequence','locus_group_id']]
pred=pred.merge(truth[['record_id','mean_log2_activity','sequence']],on='record_id',validate='many_to_one')
pred['gc_count']=pred.sequence.map(lambda s:s.count('G')+s.count('C'))
def exact_gc_auc(y,s,g):
 wins=0.;pairs=0
 for count in np.unique(g):
  pos=np.sort(s[(y==1)&(g==count)]);neg=np.sort(s[(y==0)&(g==count)])
  if not len(pos) or not len(neg):continue
  wins+=sum(np.searchsorted(neg,x,side='left')+.5*(np.searchsorted(neg,x,side='right')-np.searchsorted(neg,x,side='left')) for x in pos)
  pairs+=len(pos)*len(neg)
 return (wins/pairs if pairs else np.nan),pairs
def metrics(sub):
 y=sub.label.to_numpy(dtype=int);s=sub.score.to_numpy(dtype=float)
 g=sub.gc_count.to_numpy(dtype=int);actual=sub.mean_log2_activity.to_numpy(dtype=float)
 exact,pairs=exact_gc_auc(y,s,g)
 return {'n':len(y),'positive':int(y.sum()),'auroc':roc_auc_score(y,s),
         'auprc':average_precision_score(y,s),'spearman_activity':spearmanr(actual,s).statistic,
         'exact_gc_pair_auc':exact,'exact_gc_pairs':pairs}
rows=[]
for (cell,model),part in pred[pred.split=='test'].groupby(['cell_line','model']):
 rows.append({'cell_line':cell,'model':model,**metrics(part)})
test_metrics=pd.DataFrame(rows).sort_values(['cell_line','auroc'],ascending=[True,False])
print(test_metrics[['cell_line','model','n','auroc','auprc','exact_gc_pair_auc']].to_string(index=False))

# Resample shared genomic loci within each cell; all arms use exactly the same sampled indices.
boots=[];test=pred[pred.split=='test']
for cell in ('HepG2','K562'):
 p=test[test.cell_line==cell]
 wide=p.pivot(index='record_id',columns='model',values='score')
 base=d[(d.cell_line==cell)&(d.split=='test')].set_index('record_id').loc[wide.index]
 assert not wide.isna().any().any()
 groups=base.locus_group_id.to_numpy();group_ids=np.unique(groups)
 by_group=[np.flatnonzero(groups==g) for g in group_ids]
 truth=base.label.to_numpy(dtype=int)
 rng=np.random.default_rng(CONFIG['seed']+441+(['HepG2','K562'].index(cell)))
 for rep in range(CONFIG['bootstrap_repeats']):
  pick=rng.integers(0,len(group_ids),size=len(group_ids))
  idx=np.concatenate([by_group[k] for k in pick])
  if len(np.unique(truth[idx]))<2:continue
  aucs={model:roc_auc_score(truth[idx],wide[model].to_numpy()[idx]) for model in wide.columns}
  for model,value in aucs.items():
   boots.append({'cell_line':cell,'repeat':rep,'model':model,'auroc':value,
                 'delta_vs_spaced6_4':value-aucs['spaced6_4_logreg'],
                 'delta_vs_phase5_same_backbone':value-aucs.get('phase5_zero_shot_finetuned_'+
                   ('dnabert_legacy' if model.startswith('dnabert_legacy') else 'dnabert2'),np.nan)
                 if model in ('dnabert2_stage1_init','dnabert_legacy_stage1_init') else np.nan})
boot=pd.DataFrame(boots)
assert len(boot)>CONFIG['bootstrap_repeats']*10
for cell in ('HepG2','K562'):
 for model in ('dnabert2_stage1_init','dnabert_legacy_stage1_init'):
  v=boot[(boot.cell_line==cell)&(boot.model==model)].delta_vs_spaced6_4
  print(cell,model,'AUROC delta vs spaced6_4 95% percentile interval:',
        np.quantile(v,[.025,.5,.975]).round(4).tolist())


cell_line                                     model    n    auroc    auprc  exact_gc_pair_auc
    HepG2                       dnabert2_pretrained 2639 0.916174 0.917531           0.898704
    HepG2                      dnabert2_stage1_init 2639 0.910884 0.912718           0.892357
    HepG2                          spaced6_4_logreg 2639 0.898582 0.899496           0.880886
    HepG2                dnabert_legacy_stage1_init 2639 0.884633 0.884514           0.868094
    HepG2                                 k3_logreg 2639 0.753865 0.752798           0.720687
    HepG2         phase5_zero_shot_spaced6_4_logreg 2639 0.662802 0.670248           0.570603
    HepG2                phase5_zero_shot_k3_logreg 2639 0.652165 0.659408           0.529256
    HepG2                                 gc_logreg 2639 0.650760 0.669049           0.509986
    HepG2                phase5_zero_shot_gc_logreg 2639 0.650731 0.669062           0.509986
    HepG2                   phase5_zero_shot_raw_gc 2639 0.6

In [8]:
# 7 — Package auditable results PLUS every fitted weight, and verify both ZIPs.
RESULTS=Path('/kaggle/working/enhancer_phase6_mpra_results.zip')
COMPLETE=Path('/kaggle/working/enhancer_phase6_complete_with_weights.zip')
ART=WORK/'results';ART.mkdir(exist_ok=True)
files={}
def write_csv(name,df):
 p=ART/name;df.to_csv(p,index=False,float_format='%.12g');files[name]=p
write_csv('phase6_split.csv',d[['record_id','cell_line','locus_group_id','split_group_id','split','label']])
write_csv('phase6_validation_choices.csv',pd.DataFrame(CHOICES))
write_csv('phase6_scores_validation_and_test.csv',pred.drop(columns='sequence'))
write_csv('phase6_test_metrics.csv',test_metrics)
write_csv('phase6_locus_bootstrap.csv',boot)
expected_checkpoints={f'{cell}_{arm}_best.pt':CKPT/f'{cell}_{arm}_best.pt'
 for cell in ('HepG2','K562') for arm in ARMS[3:]}
assert len(expected_checkpoints)==6 and all(p.is_file() for p in expected_checkpoints.values())
report={'protocol_sha256':LOCK,'config':CONFIG,'input_paths':{'phase4':str(P4),'phase5':str(P5)},
 'input_sha256':{'phase4':sha_file(P4),'phase5':sha_file(P5)},
 'split_sha256':sha_file(files['phase6_split.csv']),
 'result_sha256':{name:sha_file(p) for name,p in files.items()},
 'checkpoints':{name:{'sha256':sha_file(p),'bytes':p.stat().st_size} for name,p in expected_checkpoints.items()},
 'selection':'validation AUROC, max 2 epochs; untouched test scored after selection',
 'scope':'MPRA within-design top/bottom activity quintiles; one grouped holdout; not the legacy enhancer label',
 'caveat':'Group by overlapping assay loci and exact/RC sequences; distant homologous regions may remain related.'}
report_path=ART/'phase6_provenance.json';report_path.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
files[report_path.name]=report_path
with zipfile.ZipFile(RESULTS,'w',allowZip64=True) as z:
 for name,p in files.items():z.write(p,name,compress_type=zipfile.ZIP_DEFLATED)
with zipfile.ZipFile(RESULTS) as z:
 assert z.testzip() is None
 for name,digest in report['result_sha256'].items():assert member_hash(z,name)==digest
with zipfile.ZipFile(COMPLETE,'w',allowZip64=True) as z:
 z.write(RESULTS,RESULTS.name,compress_type=zipfile.ZIP_STORED)
 for name,p in expected_checkpoints.items():z.write(p,name,compress_type=zipfile.ZIP_STORED)
 z.writestr('phase6_complete_manifest.json',json.dumps({
  'protocol_sha256':LOCK,'results_zip_sha256':sha_file(RESULTS),
  'checkpoints':report['checkpoints']},indent=2,sort_keys=True)+'\n')
with zipfile.ZipFile(COMPLETE) as z:
 assert z.testzip() is None
 assert member_hash(z,RESULTS.name)==sha_file(RESULTS)
 for name,meta in report['checkpoints'].items():assert member_hash(z,name)==meta['sha256']
print('DOWNLOAD',RESULTS,RESULTS.stat().st_size,'bytes SHA256',sha_file(RESULTS))
print('DOWNLOAD',COMPLETE,COMPLETE.stat().st_size,'bytes SHA256',sha_file(COMPLETE))
from IPython.display import FileLink,display
os.chdir('/kaggle/working')
display(FileLink(RESULTS.name))
display(FileLink(COMPLETE.name))


DOWNLOAD /kaggle/working/enhancer_phase6_mpra_results.zip 4830122 bytes SHA256 8e62c6901c2c031e3d8310935e4e11800a8aa347a04e2738f8caf677b8c1076a
DOWNLOAD /kaggle/working/enhancer_phase6_complete_with_weights.zip 2591835357 bytes SHA256 8230c322fc934a51620137e233f7a7354132597a854ea5cdef68c672e45ecba1


/kaggle/working/enhancer_phase6_mpra_results.zip

/kaggle/working/enhancer_phase6_complete_with_weights.zip